# RiemannianStaircaseOptimizer: Burer–Monteiro certification

`RiemannianStaircaseOptimizer` solves matrix-valued Rot2/Rot3 QCQPs through a low-rank Burer–Monteiro factorization, verifies an SDP dual certificate, and increases rank only when the certificate exposes a negative-curvature direction.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/certifiable/doc/RiemannianStaircaseOptimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
from gtsam.symbol_shorthand import X

## From QCQP to the staircase

The SDP variable $Z\succeq0$ is factorized as $Z=YY^\top$ with $Y\in\mathbb R^{r\times p}$. At rank $p$, the optimizer:

1. constructs `QcqpProblem(graph, p)`;
2. solves the rank-$p$ constrained problem with `AugmentedLagrangianOptimizer`;
3. builds the certificate $S=Q+\mathcal A^*(\lambda)$;
4. certifies when $S\succeq-\eta I$;
5. otherwise appends a column along the most-negative eigenvector and continues at rank $p+1$.

The current matrix QCQP traits support Rot2 and Rot3. Their solutions share a common right-orthogonal gauge, so rounded absolute rotations must be gauge-aligned before comparison.

In [3]:
num_rotations = 4
delta = 2.0 * np.pi / num_rotations
truth = [gtsam.Rot2.fromAngle(i * delta) for i in range(num_rotations)]
graph = gtsam.NonlinearFactorGraph()
initial = gtsam.Values()
for i in range(num_rotations):
    j = (i + 1) % num_rotations
    graph.add(gtsam.FrobeniusBetweenFactorRot2(
        X(i), X(j), truth[i].between(truth[j])
    ))
    perturbed = gtsam.Rot2.fromAngle(i * delta + 0.02 * i)
    initial.insert(X(i), perturbed.matrix().T)

In [4]:
alm = gtsam.AugmentedLagrangianParams()
alm.maxIterations = 80
alm.absoluteViolationTolerance = 1e-8

params = gtsam.RiemannianStaircaseParams()
params.pMin = 2
params.pMax = 4
params.eta = 1e-3
params.setAlmParams(alm)

optimizer = gtsam.RiemannianStaircaseOptimizer(graph, initial, params)
result = optimizer.optimize()
print("certified:", result.certified)
print("final rank:", result.finalRank)
print("certificate eigenvalue/bound:", result.minEigenvalue)
print("ranks visited:", result.getRanksVisited())

certified: True
final rank: 2
certificate eigenvalue/bound: -0.001
ranks visited: [2.]


`result.values` contains the final unrounded rank-$p$ matrices. When certification succeeds, `result.roundedValues()` contains the SVD projection to the intrinsic rotation rank. See `RiemannianStaircaseResult.ipynb` for diagnostics and recovery details.